# 2.0 Building the sentence table

This notebook turns article text into one row per sentence, each tagged with the company it is
about and with whether anything downstream will read it. The order is split, match aliases, flag
boilerplate, score, and then keep only what a feature consumes.

Two cells below read `mentions_other` and `is_comparative`, columns we dropped in `b1fcc6f`, so the
notebook is read rather than re-run. The stored output is the record.

The rest of the series: 2.1 works out who each sentence is about, 2.2 chooses the score, and 2.3
builds the article table.

In [1]:
import json
import time

import pandas as pd
import matplotlib.pyplot as plt

from news_sentiment.config import (
    DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    PRIMARY_TICKER,
    REPORTS_DIR,
    PROJ_ROOT,
)
from news_sentiment.text import absa, coref, entity_filter, sentiment

pd.set_option("display.max_colwidth", 200)

articles = pd.read_parquet(PROCESSED_DATA_DIR / "processed_articles.parquet")
print("corpus:", articles.shape)

before_sentences_path = INTERIM_DATA_DIR / "sentences_before_2.3.parquet"
before_articles_path = INTERIM_DATA_DIR / "articles_before_2.3.parquet"

if not before_sentences_path.exists():
    pd.read_parquet(DATA_DIR / "sentences.parquet").to_parquet(before_sentences_path, index=False)
if not before_articles_path.exists():
    pd.read_parquet(DATA_DIR / "articles.parquet").to_parquet(before_articles_path, index=False)

sentences_before = pd.read_parquet(before_sentences_path)
articles_before = pd.read_parquet(before_articles_path)
print("sentences_before:", sentences_before.shape)
print("articles_before:", articles_before.shape)

2026-08-16 11:01:06.447 | INFO     | news_sentiment.config:<module>:12 - PROJ_ROOT path is: D:\ML\stock-predictor


corpus: (2124, 8)
sentences_before: (71410, 24)
articles_before: (2124, 46)


### 1. Sentence splitting

We use spaCy's dependency-parse sentencizer rather than the rule-based one, because it reasons
about grammar and so leaves `Q3 2024.`, `U.S.`, `Inc.` and `$1.2B` intact. `MIN_SENT_CHARS = 20`
drops scraper residue.

In [4]:
text = (
    "Tesla posted Q3 2024 results, beating U.S. estimates. "
    "The company raised $1.2B in Q3. "
    "Tesla Inc. reaffirmed its guidance for the year."
)
nlp = entity_filter._get_nlp()
for s in entity_filter.split_sentences([text], nlp=nlp)[0]:
    print(repr(s))


2026-08-14 12:30:17.578 | INFO     | news_sentiment.text.entity_filter:_get_nlp:51 - Loading spaCy model 'en_core_web_sm' (excluding ner, lemmatizer)


'Tesla posted Q3 2024 results, beating U.S. estimates.'
'The company raised $1.2B in Q3.'
'Tesla Inc. reaffirmed its guidance for the year.'


Missed splits are the harder direction. Stripped HTML sometimes glues two sentences together with
no space at all, as in "...record deliveries.Musk said...", and with no whitespace token there is
nothing for the sentencizer to break on. `_fix_missing_space` inserts a space wherever a lowercase
letter or digit is followed directly by `.`, `!` or `?` and then an uppercase letter. It leaves
`U.S.` alone, because the period there comes after an uppercase letter, and it leaves `$1.2B` and
`Q3 2024.` alone, because the digits sit either side.

In [5]:
glued = "Tesla posted record deliveries.Musk said the results were strong across all regions."
entity_filter.split_sentences([glued], nlp=nlp)[0]


['Tesla posted record deliveries.',
 'Musk said the results were strong across all regions.']

We read 20 random articles to check. Counts run from 7 sentences to 43 and more, and we found no
false splits on tickers, dollar amounts or abbreviations.

One class of failure has no fix here. Scraped Yahoo transcripts weld a UI string onto the first
real sentence with no punctuation between them, as in "Oops, something went wrong Investing.com".
That is missing punctuation rather than a missing space, so neither spaCy nor our regex has
anything to catch on. We leave it, because a real fix means matching source-specific strings, and
that belongs in 1.2 rather than in a general splitter.

In [6]:
sample = articles.sample(20, random_state=7)
for _, row in sample.iterrows():
    sents = entity_filter.split_sentences([row["processed_body"]], nlp=nlp)[0]
    print(f"article {row['article_id']}: {len(row['processed_body'])} chars -> {len(sents)} sentences")
    print("   first:", sents[0][:140])


article 141078471: 3267 chars -> 27 sentences
   first: Citing Burry’s Substack ‘Trading Post,’ multiple media outlets reported that he increased his short exposure across major semiconductor and 
article 136502639: 1478 chars -> 16 sentences
   first: Oops, something went wrong Nova Sky Stories co-founder and CEO Kimbal Musk, who is also a member of Tesla's ( TSLA ) board of directors, dis
article 136933991: 1769 chars -> 15 sentences
   first: Oops, something went wrong This article first appeared on GuruFocus .
article 137897201: 3764 chars -> 34 sentences
   first: We've received your inquiry.
article 140735750: 2937 chars -> 24 sentences
   first: Michael Dell said 97% of shareholders voted to redomicile the company to Texas, calling the state “home.”
article 138590539: 545 chars -> 7 sentences
   first: Never miss a trade again with the fastest news alerts in the world!
article 140765409: 2338 chars -> 14 sentences
   first: Oops, something went wrong With short percentage of sha

article 140773279: 3154 chars -> 22 sentences
   first: Elon Musk ’s X Money is rolling out with an aggressive feature set, likely making legacy banks nervous.
article 141129157: 4821 chars -> 41 sentences
   first: We've received your inquiry.
article 138295580: 2170 chars -> 18 sentences
   first: Oops, something went wrong Investing.com --
article 139183630: 3752 chars -> 34 sentences
   first: For a stock to achieve a tenfold gain in just 10 years is a rare feat.
article 137258403: 2347 chars -> 23 sentences
   first: We've received your inquiry.
article 138452374: 2029 chars -> 21 sentences
   first: U.S. stocks traded mixed this morning, with the Dow Jones index gaining around 0.1% on Thursday.
article 141050651: 3500 chars -> 24 sentences
   first: Oops, something went wrong Well-known Tesla Inc (NASDAQ: TSLA )


article 136557730: 4265 chars -> 43 sentences
   first: U.S. stock futures are mixed as investors review Nvidia's ( NVDA ) quarterly results, which saw the AI chipmaker post better-than-expected p
article 140772782: 1533 chars -> 12 sentences
   first: Oops, something went wrong FRANKFURT, June 30 (Reuters) - Europe is an emerging but promising market for self-driving 'robotaxis', with the 
article 136888227: 1911 chars -> 23 sentences
   first: Tesla ( TSLA +3.80% ) stock has been on a big run, but it lost some steam today.
article 137258304: 3771 chars -> 37 sentences
   first: Oops, something went wrong Benzinga and Yahoo Finance LLC may earn commission or revenue on some items through the links below.
article 137930639: 1932 chars -> 13 sentences
   first: Oops, something went wrong US stocks slipped Monday to start the final three days of trading in a rollercoaster 2025 that looks likely to en


### 2. Alias matching

Matching uses a boundary-anchored regex rather than a substring test, because `"KO"` matches inside
`"Tokyo"`.

Plain `\b` is not enough on its own. It treats `-` as a non-word character, so a ticker sitting in
a URL slug like `tsla-stock-analysis` is fully delimited and matches.
`_compile_alias_pattern` uses `(?<![A-Za-z0-9-])...(?![A-Za-z0-9-])`, which treats the hyphen as
blocking too.

That guard has a cost we found later, while auditing coreference. The hyphenated forms
`"Tesla-SpaceX"` and `"SpaceX-Tesla"` are not matched as explicit mentions either, so those
sentences fall through to the coreference path. Both are genuinely about Tesla. We have not fixed
it, because widening the boundary re-opens the URL slug hole.

### 3. The company registry

`config.COMPANIES` holds 20 entries, each carrying its own alias tiers, and
`_build_ticker_patterns` selects the target's entry at runtime.

It replaced a target-relative split, where `ALIASES` held potential targets and `OTHER_COMPANIES`
held a "them" list curated from the TSLA corpus. Tesla was never added to `OTHER_COMPANIES`, so
running with `ticker="NVDA"` gave a Tesla sentence no tag at all. That is not a missing feature, it
is the pipeline quietly ceasing to work on a different ticker.

Each entry carries up to three tiers:

- **names**: unambiguous aliases. A match sets `mentions_target`.
- **person**: associated people. A match sets `mentions_ceo` and never `mentions_target`, because a
  Musk sentence may be about SpaceX or X rather than Tesla.
- **products**: model and technology names. A match sets nothing at all. The tier exists to stop the
  company name being substituted over one of its own products.

The product tier is the union across every entry rather than the target's alone. A product is never
the company under discussion, so whose product it is does not matter.

The cell below is the case the old split could not handle. Its output shows `mentions_other`, a
column we dropped in `b1fcc6f`, but the registry symmetry it demonstrates is still live.

In [8]:
tesla_under_nvda = entity_filter.tag_sentences(
    "demo_registry",
    ["Tesla shares dropped 5% after a disappointing delivery report."],
    ticker="NVDA",
)
tesla_under_nvda[["text", "mentions_target", "mentions_other"]]


,text,mentions_target,mentions_other
0,Tesla shares dropped 5% after a disappointing delivery report.,False,True


### 4. Boilerplate

`flag_boilerplate` marks any sentence whose exact text appears in at least
`BOILERPLATE_MIN_ARTICLES = 5` distinct articles. It is corpus-level by definition, so the
single-article paths leave the flag False.

In [14]:
n_comparative = int(scored_sentences["is_comparative"].sum())
n_boilerplate = int(scored_sentences["is_boilerplate"].sum())
print(f"is_comparative: {n_comparative} sentences ({n_comparative / len(scored_sentences):.2%} of corpus)")
print(f"is_boilerplate: {n_boilerplate} sentences ({n_boilerplate / len(scored_sentences):.2%} of corpus)")

scored_sentences.loc[scored_sentences["is_boilerplate"], "text"].value_counts().head(10)


is_comparative: 2259 sentences (3.16% of corpus)
is_boilerplate: 14160 sentences (19.83% of corpus)


text
All rights reserved.                                                                                                  513
Top stories, top movers, and trade ideas delivered to your inbox every weekday before and after the market closes.    450
A newsletter built for market enthusiasts by market enthusiasts.                                                      449
To add Benzinga News as your preferred source on Google, click here .                                                 407
Advertisement | Remove ads.                                                                                           332
Benzinga does not provide investment advice.                                                                          322
Invest better with The Motley Fool.                                                                                   297
Get stock recommendations, portfolio guidance, and more from The Motley Fool's premium services.                      297
Cost basis and retu

That is 14,160 sentences, 19.83% of the corpus. The ten most frequent are all legal notices,
syndication disclaimers or subscription pitches, which is what we expected: real reporting does not
repeat a full sentence verbatim across five unrelated articles, and syndicated filler always does.
Read the boilerplate line only, since the `is_comparative` line comes from the removed
other-company family.

The exact-text rule is the limitation. A publisher template that carries a filled-in variable, such
as "has had 46 moves greater than 5% over the last year", differs between articles and never
reaches the threshold, so we score it as ordinary prose. The full-corpus audit later measured that
class as a live source of error. Normalising the numbers before counting would catch it, and we
have not built it. It is also the right fix for the off-target regression that `CONF_FLOOR = 0.7`
introduces, described in 2.2 section 12.

### 5. FinBERT

We read the label order from `model.config.id2label` at load time rather than hardcoding it, so a
checkpoint that orders the labels differently still scores correctly.

In [12]:
from transformers import AutoModelForSequenceClassification
from news_sentiment.config import FINBERT_MODEL
_m = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
_m.config.id2label


D:\ML\stock-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16741.23it/s]

{0: 'positive', 1: 'negative', 2: 'neutral'}

We keep all three probabilities. `pos - neg` can be worked out at any time, but neutrality cannot
be recovered once it is thrown away, and a strongly neutral article about a recall is genuinely
different from a polarised one.

Sentences are pooled across the whole corpus into one list, deduplicated on a sha256 of the cleaned
text, sorted by length to cut padding waste, and run under `torch.no_grad()`. The cache key carries
no ticker, so any score we pay for is reused by later runs and by other tickers.

Saves merge with the cache on disk rather than replacing it. `score_headlines` once wrote only the
rows it had just scored, which overwrote around 56k sentence entries with around 2k headline ones.
The cache went from 2,101 to 23,619 entries over the run that found it. The ABSA, coreference and
judge caches all repeat the pattern rather than re-deriving it.

Aggregation is a groupby over the already-scored table and never sits inside the scoring loop, so
changing an aggregation later costs a groupby rather than a re-score. An empty sentence set
aggregates to NaN and never to 0, because a 0 would read as a measured neutral score.

`analyze` is the single-article path and calls the same three functions as the batch pipeline. One
implementation, so the demo cannot drift away from the corpus a model trains on.

In [13]:
demo_result = sentiment.analyze(
    "Tesla delivered record numbers this quarter, beating analyst estimates. "
    "The company also raised prices across its lineup. "
    "Meanwhile BYD announced an aggressive expansion into Europe.",
    ticker="TSLA",
    headline="Tesla smashes delivery estimates",
)
demo_result


2026-08-14 12:31:57.937 | INFO     | news_sentiment.text.entity_filter:_get_nlp:51 - Loading spaCy model 'en_core_web_sm' (excluding ner, lemmatizer)


2026-08-14 12:31:58.287 | INFO     | news_sentiment.text.sentiment:score_sentences:160 - score_sentences: 3 unique texts (0 cache hits, 3 to score)
2026-08-14 12:31:58.295 | INFO     | news_sentiment.text.sentiment:_load_model_and_tokenizer:77 - Loading FinBERT model 'ProsusAI/finbert'



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24453.39it/s]


Scoring sentences (FinBERT):   0%|          | 0/1 [00:00<?, ?it/s]

...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14587.24it/s]


Scoring sentences (FinBERT):   0%|          | 0/1 [00:00<?, ?it/s]


Scoring sentences (FinBERT): 100%|██████████| 1/1 [00:00<00:00, 50.93it/s]

2026-08-14 12:32:00.563 | INFO     | news_sentiment.text.sentiment:save_cache:71 - Saved sentiment cache with 2099 entries to D:\ML\stock-predictor\data\interim\finbert_cache.parquet


{'article_id': '__analyze__',
 'sent_entity_pos': 0.9476527869701385,
 'sent_entity_neg': 0.018481260165572166,
 'sent_entity_neu': 0.03386592213064432,
 'sent_entity_maxmag_pos': 0.9468124508857727,
 'sent_entity_maxmag_neg': 0.01492052897810936,
 'sent_entity_maxmag_neu': 0.03826691955327988,
 'sent_other_mean_pos': 0.9448813796043396,
 'sent_other_mean_neg': 0.012001430615782738,
 'sent_other_mean_neu': 0.04311712831258774,
 'sent_entity_lead_pos': 0.9476527869701385,
 'sent_entity_lead_neg': 0.018481260165572166,
 'sent_entity_lead_neu': 0.03386592213064432,
 'n_entity_sents': 2,
 'n_other_sents': 1,
 'n_total_sents': 3,
 'entity_share': 0.6666666666666666,
 'article_length': 180,
 'sent_headline_pos': 0.033618029206991196,
 'sent_headline_neg': 0.4334125518798828,
 'sent_headline_neu': 0.5329694151878357}

### 6. Scoring only what the features consume

    needs_score = (mentions_target | mentions_ceo) & ~is_boilerplate

Every score-derived column in `aggregate_article_features` reads one of two sentence sets.
`mentions_target` feeds the `sent_entity_*` family and `mentions_ceo` feeds `sent_ceo_*`. Anything
outside those two was being scored and then never read, including the 14,160 boilerplate sentences
from section 4, which FinBERT scored and every aggregate then excluded.

It is one predicate, imported by both the scorer and the aggregator, so the two cannot drift apart.
If a future feature reads a new sentence bucket and nobody adds it here, that feature silently gets
NaN for every article, and
`test_filtered_and_full_scoring_produce_identical_aggregates` is the alarm for exactly that.

Measured over 71,410 sentences, around 31% pass the predicate, and deduplicated on text that is a
61.5% cut in forward passes, roughly twice the speed. The article-level feature table is
bit-identical to the one the full-scoring path produced, so filtering which sentences we score
changed no feature value.

It costs us three things, all of them deliberate:

- **The sentence table is no longer fully scored.** Unscored rows carry NaN rather than 0, so
  anything that reads one by mistake fails loudly instead of averaging in a fake neutral.
- **Corpus-wide diagnostics need `only_relevant=False`**, or a sample of the unscored bucket.
- **Reproducibility has a measured floor**, covered in the next section.

### 7. The reproducibility floor, and what the table holds

Comparing the filtered run against the full-scoring baseline, 18 columns differed on a handful of
rows each by around 1e-6. The cause is not the filtering. FinBERT float32 inference depends on
batch composition, so scoring a different subset changes how sentences group into length-sorted
batches, which changes the padding. We confirmed it by re-running with the cache pre-seeded so both
paths saw identical probabilities, and they then matched bit-exactly.

The maximum absolute difference across the 21,520 shared hashes is 2.44e-6, with nothing above
1e-4. Compare feature tables across runs with an absolute tolerance of around 1e-5, not with
equality and not with a relative tolerance, because 1e-6 on a mean of 0.02 is a large relative
delta and that is what tripped our first check.

The output is `SENTENCE_COLUMNS`, one row per sentence:

```
article_id, sent_idx, text, mentions_target, mentions_ceo,
resolved_by_coref, is_boilerplate, char_len,
mention_char_start, mention_char_end
```

Nothing above sets `resolved_by_coref` or the two span columns. Those come from the coreference
path, which is notebook 2.1.